# Table S9. Other HCW Role Composition

This notebook generates the supplementary breakdown of Other HCW-initiated threads by the structured role category of the thread-initiating sender.


## Inputs and scope

The notebook reads the stable messages_with_threads.parquet and analysis_df_all_exposures_with_encounter_covariates.parquet handoffs produced by notebooks 02 and 05. It does not reconstruct exposures, alter the analytic sample, or fit statistical models.

Outputs are written to a timestamped directory under outputs/runs/ by default.


In [ ]:
from pathlib import Path
from datetime import datetime
import os

import pandas as pd
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(
    os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")
).expanduser().resolve()
WORKING_DATA_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()
RUN_DIR = (
    OUTPUT_ROOT
    / "runs"
    / f"table_s9_other_hcw_role_composition_{datetime.now():%Y%m%d_%H%M%S}"
)
TABLE_DIR = RUN_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DF_PATH = (
    WORKING_DATA_DIR
    / "analysis_df_all_exposures_with_encounter_covariates.parquet"
)
MESSAGES_WITH_THREADS_PATH = (
    WORKING_DATA_DIR
    / "messages_with_threads.parquet"
)

if not ANALYSIS_DF_PATH.exists():
    raise FileNotFoundError(
        f"Missing analysis handoff: {ANALYSIS_DF_PATH}\n"
        "Run notebooks 01 through 05 first."
    )

if not MESSAGES_WITH_THREADS_PATH.exists():
    raise FileNotFoundError(
        f"Missing threaded-message handoff: {MESSAGES_WITH_THREADS_PATH}\n"
        "Run notebooks 01 and 02 first."
    )

analysis_df = pd.read_parquet(ANALYSIS_DF_PATH, engine="pyarrow")
message_schema_columns = pd.read_parquet(
    MESSAGES_WITH_THREADS_PATH,
    engine="pyarrow",
).columns
message_columns = ["MESSAGE_ID", "SENDER_CATEGORY"]
if "SENDER_TYPE" in message_schema_columns:
    message_columns.append("SENDER_TYPE")
elif "Sender_all" in message_schema_columns:
    message_columns.append("Sender_all")

all_messages_clean = pd.read_parquet(
    MESSAGES_WITH_THREADS_PATH,
    columns=message_columns,
    engine="pyarrow",
)
if "SENDER_TYPE" not in all_messages_clean.columns:
    all_messages_clean["SENDER_TYPE"] = all_messages_clean.get(
        "Sender_all",
        all_messages_clean["SENDER_CATEGORY"],
    )


In [ ]:
ROLE_TABLE_TITLE = (
    "Table S9. Composition and 7-Day Patient Reply Rates of "
    "Other HCW-Initiated Threads by Role"
)

ROLE_TABLE_NOTE = (
    "Role categories were derived from the structured provider-role mapping "
    "used to define the physician/APP boundary. The physician category "
    "included physicians, residents, fellows, anesthesiologists, and doctors "
    "of osteopathic medicine. The APP category included nurse practitioners, "
    "nurse midwives, certified registered nurse anesthetists, physician "
    "assistants, advanced practice providers, and clinical nurse specialists. "
    "Other HCW comprised all remaining nonpatient, nonsystem health care "
    "worker roles. Percentages use all Other HCW-initiated threads as the "
    "denominator. Patient reply was defined as the first patient reply within "
    "7 days after thread initiation. HCW indicates health care worker; "
    "APP, advanced practice provider."
)



In [ ]:
required_thread_columns = {
    "thread_id",
    "thread_first_message_id",
    "thread_initiator_group",
    "patient_replied_7d",
}

required_message_columns = {
    "MESSAGE_ID",
    "SENDER_CATEGORY",
    "SENDER_TYPE",
}

missing_thread_columns = required_thread_columns - set(analysis_df.columns)
missing_message_columns = required_message_columns - set(all_messages_clean.columns)

if missing_thread_columns:
    raise KeyError(
        "analysis_df is missing required columns: "
        + ", ".join(sorted(missing_thread_columns))
    )

if missing_message_columns:
    raise KeyError(
        "all_messages_clean is missing required columns: "
        + ", ".join(sorted(missing_message_columns))
    )

thread_sample = analysis_df.copy()

if thread_sample["thread_id"].duplicated().any():
    raise AssertionError(
        "analysis_df must contain one row per HCW-initiated thread."
    )

thread_sample["thread_first_message_id"] = (
    thread_sample["thread_first_message_id"].astype("string").str.strip()
)

message_role_lookup = all_messages_clean[
    ["MESSAGE_ID", "SENDER_CATEGORY", "SENDER_TYPE"]
].copy()

message_role_lookup["MESSAGE_ID"] = (
    message_role_lookup["MESSAGE_ID"].astype("string").str.strip()
)

message_role_lookup = (
    message_role_lookup.drop_duplicates("MESSAGE_ID", keep="first")
    .rename(
        columns={
            "MESSAGE_ID": "thread_first_message_id",
            "SENDER_CATEGORY": "initiator_role_bucket",
            "SENDER_TYPE": "initiator_detailed_role",
        }
    )
)

other_hcw_threads = (
    thread_sample.loc[
        thread_sample["thread_initiator_group"].eq("OtherHuman"),
        [
            "thread_id",
            "thread_first_message_id",
            "patient_replied_7d",
        ],
    ]
    .merge(
        message_role_lookup,
        on="thread_first_message_id",
        how="left",
        validate="many_to_one",
    )
)

other_hcw_threads["initiator_role_bucket"] = (
    other_hcw_threads["initiator_role_bucket"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Unknown")
)

other_hcw_threads["patient_replied_7d"] = pd.to_numeric(
    other_hcw_threads["patient_replied_7d"],
    errors="raise",
).astype(int)

unexpected_roles = set(other_hcw_threads["initiator_role_bucket"].dropna()) & {
    "Physician",
    "APP",
    "Patient",
    "System",
}

if unexpected_roles:
    raise AssertionError(
        "Roles outside the Other HCW definition were found: "
        + ", ".join(sorted(unexpected_roles))
    )

n_other_hcw_threads = len(other_hcw_threads)

if n_other_hcw_threads == 0:
    raise ValueError("No Other HCW-initiated threads were found.")

role_summary = (
    other_hcw_threads.groupby(
        "initiator_role_bucket",
        dropna=False,
        observed=True,
    )
    .agg(
        number_of_threads=("thread_id", "nunique"),
        patient_replied_7d_n=("patient_replied_7d", "sum"),
    )
    .reset_index()
    .rename(columns={"initiator_role_bucket": "Role"})
)

role_summary["percentage_of_other_hcw_threads"] = (
    100 * role_summary["number_of_threads"] / n_other_hcw_threads
)

role_summary["patient_replied_7d_pct"] = (
    100
    * role_summary["patient_replied_7d_n"]
    / role_summary["number_of_threads"]
)

role_summary = (
    role_summary.sort_values(
        ["number_of_threads", "Role"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

if role_summary["number_of_threads"].sum() != n_other_hcw_threads:
    raise AssertionError(
        "Role-specific thread counts do not sum to all Other HCW threads."
    )

if not abs(role_summary["percentage_of_other_hcw_threads"].sum() - 100) < 1e-8:
    raise AssertionError("Role percentages do not sum to 100%.")

other_hcw_role_table = pd.DataFrame(
    {
        "Role": role_summary["Role"],
        "Number of HCW-initiated threads": role_summary[
            "number_of_threads"
        ].map(lambda value: f"{int(value):,}"),
        "Percentage of Other HCW threads": role_summary[
            "percentage_of_other_hcw_threads"
        ].map(lambda value: f"{value:.1f}%"),
        "Patient reply within 7 days, n (%)": [
            f"{int(n):,} ({pct:.1f}%)"
            for n, pct in zip(
                role_summary["patient_replied_7d_n"],
                role_summary["patient_replied_7d_pct"],
            )
        ],
    }
)

total_reply_n = int(other_hcw_threads["patient_replied_7d"].sum())
total_reply_pct = 100 * total_reply_n / n_other_hcw_threads

total_row = pd.DataFrame(
    [
        {
            "Role": "All Other HCW roles",
            "Number of HCW-initiated threads": f"{n_other_hcw_threads:,}",
            "Percentage of Other HCW threads": "100.0%",
            "Patient reply within 7 days, n (%)": (
                f"{total_reply_n:,} ({total_reply_pct:.1f}%)"
            ),
        }
    ]
)

other_hcw_role_table = pd.concat(
    [other_hcw_role_table, total_row],
    ignore_index=True,
)



In [ ]:
ROLE_TABLE_XLSX = TABLE_DIR / "table_S9_other_hcw_role_composition.xlsx"
ROLE_TABLE_HTML = TABLE_DIR / "table_S9_other_hcw_role_composition.html"
ROLE_TABLE_CSV = TABLE_DIR / "table_S9_other_hcw_role_composition.csv"

other_hcw_role_table.to_csv(ROLE_TABLE_CSV, index=False)

with pd.ExcelWriter(ROLE_TABLE_XLSX, engine="openpyxl") as writer:
    other_hcw_role_table.to_excel(
        writer,
        sheet_name="Table S9",
        startrow=2,
        index=False,
    )

    worksheet = writer.book["Table S9"]

    worksheet.merge_cells("A1:D1")
    worksheet["A1"] = ROLE_TABLE_TITLE
    worksheet["A1"].font = Font(name="Arial", size=10, bold=True)
    worksheet["A1"].alignment = Alignment(wrap_text=True, vertical="top")

    header_row = 3

    for cell in worksheet[header_row]:
        cell.font = Font(name="Arial", size=10, bold=True)
        cell.fill = PatternFill(fill_type="solid", fgColor="D9E1F2")
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    final_data_row = header_row + len(other_hcw_role_table)
    thin_border = Side(style="thin", color="000000")

    for cell in worksheet[header_row]:
        cell.border = Border(top=thin_border, bottom=thin_border)

    for cell in worksheet[final_data_row]:
        cell.font = Font(name="Arial", size=10, bold=True)
        cell.border = Border(top=thin_border, bottom=thin_border)

    note_row = final_data_row + 2
    worksheet.merge_cells(
        start_row=note_row,
        start_column=1,
        end_row=note_row,
        end_column=4,
    )
    worksheet.cell(row=note_row, column=1, value=f"Note. {ROLE_TABLE_NOTE}")
    worksheet.cell(row=note_row, column=1).font = Font(name="Arial", size=10)
    worksheet.cell(row=note_row, column=1).alignment = Alignment(
        wrap_text=True,
        vertical="top",
    )

    worksheet.column_dimensions["A"].width = 30
    worksheet.column_dimensions["B"].width = 24
    worksheet.column_dimensions["C"].width = 25
    worksheet.column_dimensions["D"].width = 32
    worksheet.row_dimensions[1].height = 30
    worksheet.row_dimensions[note_row].height = 90
    worksheet.freeze_panes = "A4"
    worksheet.sheet_view.showGridLines = False

role_table_styled = (
    other_hcw_role_table.style.hide(axis="index")
    .set_properties(subset=["Role"], **{"text-align": "left"})
    .set_properties(
        subset=[
            "Number of HCW-initiated threads",
            "Percentage of Other HCW threads",
            "Patient reply within 7 days, n (%)",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("border-top", "1.5px solid black"),
                    ("border-bottom", "1px solid black"),
                    ("font-weight", "bold"),
                    ("padding", "6px"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "none"), ("padding", "6px")],
            },
            {
                "selector": "tbody tr:last-child td",
                "props": [
                    ("border-top", "1px solid black"),
                    ("border-bottom", "1.5px solid black"),
                    ("font-weight", "bold"),
                ],
            },
        ]
    )
)

role_table_html = (
    f"<p><strong>{ROLE_TABLE_TITLE}</strong></p>"
    + role_table_styled.to_html()
    + f"<p><em>Note.</em> {ROLE_TABLE_NOTE}</p>"
)

ROLE_TABLE_HTML.write_text(role_table_html, encoding="utf-8")

print("Read analysis handoff:", ANALYSIS_DF_PATH)
print("Read threaded messages:", MESSAGES_WITH_THREADS_PATH)
print("Other HCW-initiated threads:", f"{n_other_hcw_threads:,}")
print("Saved:", ROLE_TABLE_XLSX)
print("Saved:", ROLE_TABLE_HTML)
print("Saved:", ROLE_TABLE_CSV)
